In [1]:
# 第一步：读入所有chunk，计算该问题所有chunk的分数，以JSON形式保存到本地；如果文件存在，则直接读取
# 第二步：取top20计算IDCG
# 第三步：用bm25、vdb和rrf分别召回top20，计算DCG，得到NDCG，输出比较性能。

In [2]:
from bm25 import BM25Retriever
from vdb import FaissRetriever
from rrf import rrf_fusion
from langchain_ollama import ChatOllama
from typing import Annotated, TypedDict, List, Union, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from operator import add
from langgraph.graph import StateGraph, END, START
from sentence_transformers import CrossEncoder
from langchain_core.prompts import ChatPromptTemplate

d:\Miniconda\envs\agent\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
import torch
print(f'{torch.cuda.is_available()} {torch.cuda.device_count()} {torch.cuda.get_device_name(0)}')

True 1 NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
chunk_path = 'first10_chunks.json'
bm25_retriever = BM25Retriever(chunks_file=chunk_path)
faiss_retriever = FaissRetriever(chunks_file=chunk_path)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device='cuda')
llm = ChatOllama(model='qwen3:8b', num_ctx=4096, temperature=0, format='json')

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\DIVINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.704 seconds.
Prefix dict has been built successfully.


已加载 FAISS 索引，包含 62 个向量。


In [5]:
import os
import json
import numpy as np

def get_all_chunk_scores(query: str, chunks_file: str, cache_file: str):
    # 如果缓存存在，直接读取（宁静且高效）
    if os.path.exists(cache_file):
        print(f"[INFO] 从缓存加载全量分数: {cache_file}")
        with open(cache_file, 'r', encoding='utf-8') as f:
            return json.load(f)

    # 否则，加载所有数据并计算
    with open(chunks_file, 'r', encoding='utf-8') as f:
        all_chunks = json.load(f)
    
    print(f"[INFO] 开始暴力计算 {len(all_chunks)} 个 Chunk 的相关性分数...")
    texts = [c['chunk_text'] for c in all_chunks]
    queries = [query] * len(texts)
    
    # 这里的 reranker 沿用你代码中的 CrossEncoder
    scores = reranker.predict(list(zip(queries, texts)))
    
    # 合并结果
    scored_chunks = []
    for chunk, score in zip(all_chunks, scores):
        chunk['gold_score'] = float(score) # 转换为 float 方便 JSON 序列化
        scored_chunks.append(chunk)
    
    # 保存结果
    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(scored_chunks, f, ensure_ascii=False, indent=2)
    
    return scored_chunks

In [9]:
def calculate_dcg(rel_scores, k=20):
    rel_scores = np.asfarray(rel_scores)[:k]
    if rel_scores.size:
        return np.sum(rel_scores / np.log2(np.arange(2, rel_scores.size + 2)))
    return 0.0

def evaluate_retrievers(query, scored_chunks_all):
    # 1. 计算 IDCG (基于上帝视角的 Top 20)
    ideal_sorted = sorted(scored_chunks_all, key=lambda x: x['gold_score'], reverse=True)
    idcg = calculate_dcg([x['gold_score'] for x in ideal_sorted], k=20)
    
    # 构建快速查找映射表 {chunk_id: score}
    score_map = {f"{c['chpt_id']}_{c['chunk_id']}": c['gold_score'] for c in scored_chunks_all}

    # 2. 定义各个检索实验
    experiments = {
        "FAISS (VDB)": faiss_retriever.retrieve(query, top_k=20),
        "BM25": bm25_retriever.retrieve(query, top_k=20),
        "RRF Fusion": rrf_fusion([
            faiss_retriever.retrieve(query, top_k=20),
            bm25_retriever.retrieve(query, top_k=20)
        ], k=30, top_k=20)
    }

    print(f"\n{'Retriever':<15} | {'DCG@20':<10} | {'NDCG@20':<10}")
    print("-" * 45)

    for name, results in experiments.items():
        # 获取检索结果在上帝视角下的得分
        retrieved_scores = [score_map.get(f"{r['chpt_id']}_{r['chunk_id']}", -10) for r in results]
        dcg = calculate_dcg(retrieved_scores, k=20)
        ndcg = dcg / idcg if idcg > 0 else 0
        
        print(f"{name:<15} | {dcg:<10.4f} | {ndcg:<10.4f}")

# --- 主程序执行 ---
query_text = "杨奇从城主府偷走了什么？"
cache_path = f"{query_text}_{chunk_path}"

# 步骤 1
all_scored = get_all_chunk_scores(query_text, chunk_path, cache_path)

# 步骤 2 & 3
evaluate_retrievers(query_text, all_scored)

[INFO] 开始暴力计算 62 个 Chunk 的相关性分数...

Retriever       | DCG@20     | NDCG@20   
---------------------------------------------
FAISS (VDB)     | 4.4608     | 0.7624    
BM25            | 3.1156     | 0.5325    
RRF Fusion      | 4.0379     | 0.6902    
